# FactorCake
### Tugas 1 GenAI LLM: Michelle C14240169

Saya membuat Chrome extension yang dapat mendetect sebuah media berupa text based ataupun video based yang berisikan informasi dan memberikan output berupa persentase fakta dan hoax serta analisis yang dihasilkan oleh LLM. Saya menggunakan model LLM
Qwen/Qwen2.5-7B-Instruct.

In [2]:
!pip install fastapi uvicorn pyngrok nest-asyncio transformers torch accelerate youtube-transcript-api==0.6.3 yt-dlp
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [3]:
from google.colab import userdata
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn
import threading
import traceback
import json
import torch
import os
import tempfile
from transformers import pipeline
from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp

In [4]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ---------------------------------------------------------
# 1. Model Setup: Qwen LLM & Whisper Speech-to-Text
# ---------------------------------------------------------
# https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# for input and output (text based)
# pipeline: https://huggingface.co/docs/transformers/v5.17.0/en/main_classes/pipelines#transformers.TextGenerationPipeline
print("Loading Qwen 2.5 LLM pipeline...")
classifier = pipeline(
    "text-generation",
    model=LLM_MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# for translating video sound (transcript) to text
# pipeline: https://huggingface.co/docs/transformers/v5.17.0/en/main_classes/pipelines#transformers.AutomaticSpeechRecognitionPipeline
# model: https://huggingface.co/openai/whisper-base
print("Loading Whisper Speech-to-Text pipeline...")
stt_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
except Exception as e:
    print(f"Error fetching Ngrok token: {e}")

# ---------------------------------------------------------
# 2. Core Helper Functions
# ---------------------------------------------------------
def analyze_text_with_llm(text_to_check: str):
    messages = [
        {
            "role": "system",
            "content": "You are a fact-checking assistant. Always respond in raw, valid JSON."
        },
        {
            "role": "user",
            "content": f"""Analyze the following text/transcript for factual accuracy and potential hoaxes.
Text to analyze:
"{text_to_check[:12000]}"

Respond STRICTLY in valid JSON with these exact keys:
{{
  "factual_percentage": integer (0-100),
  "hoax_percentage": integer (0-100),
  "analysis": "A concise 2-3 sentence explanation."
}}"""
        }
    ]

    outputs = classifier(
        messages,
        max_new_tokens=256,
        temperature=0.1,
        return_full_text=False
    )

    raw_text = outputs[0]["generated_text"].strip()
    if raw_text.startswith("```"):
        raw_text = raw_text.split("```")[1]
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]
    return json.loads(raw_text.strip())

def transcribe_audio_from_url(video_url: str):
    """Helper function to download audio and transcribe via Whisper with long-form support."""
    temp_dir = tempfile.mkdtemp()
    output_audio_path = os.path.join(temp_dir, "audio.mp3")

    try:
        ydl_opts = {
            'format': 'm4a/bestaudio/best',
            'outtmpl': os.path.join(temp_dir, 'audio.%(ext)s'),
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'quiet': True,
            'no_warnings': True,
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])

        # Added return_timestamps=True to handle videos longer than 30 seconds
        stt_result = stt_pipeline(output_audio_path, return_timestamps=True)
        return stt_result["text"].strip()
    finally:
        if os.path.exists(output_audio_path):
            os.remove(output_audio_path)

# ---------------------------------------------------------
# 3. Route: Webpage Text Analysis
# ---------------------------------------------------------
@app.post("/fact-check-text")
async def fact_check_text(request: Request):
    try:
        data = await request.json()
        text_to_check = data.get("text", "")
        if not text_to_check.strip():
            return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": "No text found."}
        return analyze_text_with_llm(text_to_check)
    except Exception as e:
        return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": f"Backend Error: {str(e)}"}

# ---------------------------------------------------------
# 4. Route: YouTube Video (Transcript API + Whisper Fallback)
# ---------------------------------------------------------
@app.post("/fact-check-youtube")
async def fact_check_youtube(request: Request):
    try:
        data = await request.json()
        video_id = data.get("video_id", "")
        video_url = f"https://www.youtube.com/watch?v={video_id}"

        if not video_id:
            return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": "No video ID provided."}

        transcript_text = ""

        # Try fetching official transcripts first
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcript_list:
                if t.language_code in ['id', 'en']:
                    transcript_text = " ".join([item['text'] for item in t.fetch()])
                    break
            if not transcript_text:
                transcript_text = " ".join([item['text'] for item in next(iter(transcript_list)).fetch()])
        except Exception:
            pass # Transcripts disabled, will use Whisper fallback below

        # Fallback to Whisper audio transcription if YouTube captions are disabled
        if not transcript_text.strip():
            print("YouTube captions disabled. Falling back to Whisper audio transcription...")
            transcript_text = transcribe_audio_from_url(video_url)

        if not transcript_text.strip():
            return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": "Could not retrieve transcript or audio speech."}

        return analyze_text_with_llm(transcript_text)

    except Exception as e:
        traceback.print_exc()
        return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": f"YouTube Error: {str(e)}"}

# ---------------------------------------------------------
# 5. Route: Generic Video Audio Extraction & STT
# ---------------------------------------------------------
@app.post("/fact-check-video")
async def fact_check_video(request: Request):
    try:
        data = await request.json()
        video_url = data.get("url", "")

        if not video_url:
            return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": "No video URL provided."}

        print(f"Extracting audio from generic URL: {video_url}")
        transcript_text = transcribe_audio_from_url(video_url)

        if not transcript_text:
            return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": "No spoken dialogue detected in video."}

        return analyze_text_with_llm(transcript_text)

    except Exception as e:
        traceback.print_exc()
        return {"factual_percentage": 0, "hoax_percentage": 0, "analysis": f"Video Processing Error: {str(e)}"}

# ---------------------------------------------------------
# Start Server Tunnel
# ---------------------------------------------------------
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(8000)
print(f"\nYour API is live at: {tunnel.public_url}\n")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()

Loading Qwen 2.5 LLM pipeline...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading Whisper Speech-to-Text pipeline...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]


Your API is live at: https://royal-backlit-duct.ngrok-free.dev

